In [9]:
# Libraries
import os
import joblib
import numpy as np
import pandas as pd
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

# Load CSVs
train_df = pd.read_csv('../backup/data/sign_mnist_train.csv')
test_df  = pd.read_csv('../backup/data/sign_mnist_test.csv')

y_train = train_df['label'].values
X_train_raw = train_df.drop('label', axis=1).values
y_test = test_df['label'].values
X_test_raw = test_df.drop('label', axis=1).values

# Reshape into images
X_train_images = X_train_raw.reshape(-1, 28, 28)
X_test_images  = X_test_raw.reshape(-1, 28, 28)

# HOG params (keep these consistent everywhere)
hog_params = {
    'orientations': 12,
    'pixels_per_cell': (4, 4),
    'cells_per_block': (2, 2),
    'block_norm': 'L2-Hys'
}

def compute_hog_features(images, **hp):
    features = [hog(img, orientations=hp['orientations'],
                         pixels_per_cell=hp['pixels_per_cell'],
                         cells_per_block=hp['cells_per_block'],
                         block_norm=hp['block_norm']) for img in images]
    return np.array(features)

X_train_hog = compute_hog_features(X_train_images, **hog_params)
X_test_hog  = compute_hog_features(X_test_images,  **hog_params)

# Feature scaling (fit on train, apply to test)
scaler = StandardScaler()
X_train_hog_scaled = scaler.fit_transform(X_train_hog)
X_test_hog_scaled  = scaler.transform(X_test_hog)

# Train model (use scaled features)
model = RandomForestClassifier(n_estimators=500, max_depth=30,
                               random_state=42, n_jobs=-1)
model.fit(X_train_hog_scaled, y_train)

# Evaluate
y_pred = model.predict(X_test_hog_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

# Save everything together (use consistent path)
save_path = '../models'
os.makedirs(save_path, exist_ok=True)
joblib.dump({
    'model': model,
    'scaler': scaler,
    'hog_params': hog_params
}, os.path.join(save_path, 'sign_model_500_30_HOG12.pkl'))


Accuracy: 0.9336307863915226
Confusion matrix:
 [[331   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0]
 [  0 411   0   0   0   0   0   0   0  20   0   0   0   0   0   0   0   0
    0   0   0   1   0   0]
 [  0   0 310   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0]
 [  0   0   0 232   0   0   0   0   0   0   0   0   0   0   0   0   0  13
    0   0   0   0   0   0]
 [  0   0   0   0 498   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0]
 [  0   0   0   0   0 247   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0]
 [  0   0   0   0   0   0 317  14   0   0   0   0   0   0   0   0   0   0
    7   0   0   0  10   0]
 [  0   0   0   0   0   0  20 416   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0]
 [  0   1   0   0   0   0   0   0 270   0   0   0   0   0   0   0   0  13
    1   0   0   0   0   3]
 [  0   0   0   0   0   0   0  14   0 232  

['../models/sign_model_500_30_HOG12.pkl']